# **import libraries**

In [7]:
import json 
import os
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

# **Load Clean Dataset**

In [14]:
base_dir = os.path.dirname(os.getcwd())
main_path = os.path.join(base_dir,"data","arxiv_cs.json")

with open(main_path,"r",encoding="utf-8") as f:
    papers = json.load(f)

print("Total Paper's ",len(papers))

Total Paper's  60000


# **Extract text**

In [18]:
texts = []

for paper in papers:
    text = paper.get("abstract")
    if text and text.strip():
        texts.append(text)

print("Valid texts:",len(texts))

Valid texts: 60000


# **Create Embeddings**

In [19]:
embeddings = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
)

c:\Users\shriyansh\OneDrive\Desktop\GenAI-main\ResearchMind_AI_Chatbot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# **Create FAISS Vector Store**

In [20]:
vectorstore = FAISS.from_texts(texts[:5000],embeddings) # Use Subset for notebook

print("FAISS Index Created")

FAISS Index Created


# **Retrieve Similar Documents**

In [22]:
query = "what is machine learning?"

docs = vectorstore.similarity_search(query,k=3)

for i, doc in enumerate(docs):
    print(f"\n --- Document {i+1} ---")
    print(doc.page_content[:500])


 --- Document 1 ---
  Many problems that arise in machine learning domain deal with nonlinearity
and quite often demand users to obtain global optimal solutions rather than
local optimal ones. Optimization problems are inherent in machine learning
algorithms and hence many methods in machine learning were inherited from the
optimization literature. Popularly known as the initialization problem, the
ideal set of parameters required will significantly depend on the given
initialization values. The recently developed 

 --- Document 2 ---
  Knowledge could be gained from experts, specialists in the area of interest,
or it can be gained by induction from sets of data. Automatic induction of
knowledge from data sets, usually stored in large databases, is called data
mining. Data mining methods are important in the management of complex systems.
There are many technologies available to data mining practitioners, including
Artificial Neural Networks, Regression, and Decision Trees. Neural ne

# **Setup LLM(Groq)**

In [27]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv

load_dotenv()

llm = ChatGroq(
    groq_api_key=os.getenv("GROQ_API_KEY"),
    model_name = "llama-3.1-8b-instant"
)

# **Create Prompt**

In [29]:
from langchain.prompts import PromptTemplate

prompt = PromptTemplate(
    input_variables=["context","question"],
    template="""
You are an expert AI research assitant.

Context:
{context}

Question:
{question}

Answer clearly and simply
"""
)

# **Build RAG Chain**

In [30]:
from langchain.chains.retrieval_qa.base import RetrievalQA

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff",
    chain_type_kwargs= {"prompt":prompt}
)

# **Ask Questions**

In [31]:
query = "Explain Deep Learning in simple terms"

answer = qa_chain.run(query)

print(answer)

c:\Users\shriyansh\OneDrive\Desktop\GenAI-main\ResearchMind_AI_Chatbot\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:141: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 0.3.0. Use invoke instead.
  warn_deprecated(


**Deep Learning in Simple Terms:**

Deep Learning is a type of Artificial Intelligence (AI) that allows computers to learn and improve on their own by analyzing large amounts of data. It's like teaching a child to recognize pictures of cats and dogs.

Imagine you show a child a picture of a cat and say, "This is a cat." You repeat this process many times, showing the child different pictures of cats and saying, "This is a cat." After a while, the child starts to recognize what a cat looks like, even if it's a different picture.

Deep Learning is similar, but instead of a child, it's a computer that's learning to recognize patterns in data. The computer looks at many examples of data, like pictures of cats, and learns to identify what makes them similar or different.

There are three main parts to Deep Learning:

1. **Neural Networks**: These are computer systems that mimic the way the human brain works. They have many layers, like a big sandwich, and each layer processes the informatio

In [32]:
questions = [
    "What is NLP?",
    "Explain neural networks",
    "What is reinforcement learning?"
]

for q in questions:
    print("\nQuestion: ",q)
    print("Answer: ",qa_chain.run(q))


Question:  What is NLP?
Answer:  NLP stands for Natural Language Processing. It's a field of research that focuses on developing computer systems that can understand, interpret, and generate human language, such as text or speech.

Question:  Explain neural networks
Answer:  **What are Neural Networks?**

A neural network is a computer program that's designed to mimic the way our brains work. It's a network of connected nodes or "neurons" that process and transmit information.

**How Do Neural Networks Work?**

Here's a simplified explanation:

1. **Input**: The network receives data, like numbers or words.
2. **Processing**: Each node in the network takes the input data, performs a calculation, and sends the result to other nodes.
3. **Connections**: The nodes are connected in a way that allows them to share information and pass it on to other nodes.
4. **Output**: The final nodes in the network produce a result, like a prediction or a decision.

**Key Features of Neural Networks**

